# AI Engineering Interview Prep
## Section: Fine-Tuning & Model Adaptation

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


---
## 🔧 Section 5 — Fine-Tuning and Model Adaptation

### Q1. What is fine-tuning, and when should you fine-tune an LLM?
**A:** Fine-tuning = continuing training a pre-trained model on a smaller, task-specific dataset to specialise its behaviour.

**When to fine-tune (not just prompt):**
1. Consistent style/format the model can't learn from prompts alone
2. Domain-specific vocabulary and reasoning patterns (legal, medical, code)
3. Reducing prompt length — "bake" long instructions into model weights
4. High-volume tasks where a fine-tuned 7B beats a prompted 70B (cost savings)
5. Privacy — don't want sensitive prompts sent to third-party APIs

**When NOT to fine-tune:**
- Knowledge that changes frequently (use RAG instead)
- When you don't have quality labelled data (>500-1000 examples minimum)
- When prompt engineering can get you 90% of the way there

🏷️ *I haven't needed to fine-tune for Nyaya-Pro because RAG + prompt engineering achieved the required quality. Fine-tuning would add weeks of data prep for marginal gains.*


### Q2. Explain the difference between full fine-tuning and PEFT (Parameter-Efficient Fine-Tuning).
**A:**
**Full fine-tuning:** Update ALL model parameters on your dataset. Best quality but requires enormous GPU memory (the entire model in FP32 for gradients + optimiser states). A 7B model needs ~112GB GPU RAM. Expensive, slow, and risks catastrophic forgetting.

**PEFT:** Only update a small fraction of parameters (adapters, prefixes, or low-rank matrices) while keeping base model weights frozen.

| | Full FT | LoRA | QLoRA |
|--|---------|------|-------|
| Memory | ~112GB (7B) | ~16GB (7B) | ~6GB (7B) |
| Quality | Highest | Near-full | Slightly lower |
| Speed | Slow | Fast | Fast |
| Hardware | 8×A100 | 1×RTX 3090 | 1×RTX 2080 |

PEFT (especially LoRA/QLoRA) is the standard for AI engineering — you get 90-95% of full FT quality at 10% of the cost.


### Q3. What is LoRA (Low-Rank Adaptation), and how does it work?
**A:** LoRA freezes all original model weights and injects small trainable matrices into the attention layers.

For a weight matrix W (shape d×d), instead of updating W directly, LoRA learns:
```
ΔW = A × B    where A: d×r, B: r×d, and r << d (e.g., r=8, d=4096)
```
Training only A and B — the actual weight update is `W + ΔW = W + A×B`.

Number of trainable parameters: `2 × r × d` instead of `d²`. For d=4096, r=8: 65,536 vs 16.7M — 256× fewer params to train.

At inference: merge A×B into W once (zero overhead) or keep separate (allows hot-swapping adapters).

🏷️ *If I fine-tune a legal LLM variant of Nyaya-Pro, LoRA on Llama 3 8B would be my first choice — can train on a single A100 with custom legal QA pairs.*


### Q4. What is QLoRA, and how does it enable fine-tuning on consumer hardware?
**A:** QLoRA = Quantised LoRA. Takes LoRA and adds 4-bit NF4 (NormalFloat4) quantisation of the base model.

Three key innovations:
1. **4-bit NF4 quantisation** — base model weights stored in 4-bit (not FP16), reducing memory 4×
2. **Double quantisation** — quantise the quantisation constants too, saving another ~0.4 bits/param
3. **Paged optimisers** — manage GPU memory spikes during training via CPU offloading

Result: A 65B parameter model fits on a single 48GB GPU. A 7B model fits on a single 16GB consumer GPU (RTX 4080).

Trade-off: Slightly slower training than LoRA (quantise/dequantise overhead) and marginally lower quality than LoRA on full precision. But makes fine-tuning accessible to developers without enterprise GPU clusters.


### Q5. Explain Prefix Tuning and Prompt Tuning. How are they different from LoRA?
**A:**
**Prompt Tuning:** Learns a set of soft, continuous embedding vectors prepended to every input. These "virtual tokens" aren't readable text — they're optimised directly in embedding space. Only these prefix embeddings are trained; model weights frozen.

**Prefix Tuning:** Similar but adds trainable vectors to ALL layers' K and V matrices, not just the input embeddings. More expressive than prompt tuning.

**vs LoRA:**
| | LoRA | Prompt Tuning | Prefix Tuning |
|--|------|--------------|---------------|
| What's trained | Low-rank matrices in attention | Input soft tokens | K,V prefix in every layer |
| Parameters | ~0.1-1% | Very few (~0.01%) | Few (~0.01-0.1%) |
| Interpretability | None (weight changes) | None (continuous vectors) | None |
| Quality | Best | Lower | Medium |
| Inference overhead | None (after merging) | Small | Small per layer |


### Q6. What is adapter-based fine-tuning?
**A:** Adapters are small bottleneck neural network modules inserted between Transformer layers. The original model weights are frozen; only adapter parameters are trained.

Architecture (per layer):
```
Original Output → Down-project (d→r) → Activation → Up-project (r→d) → Add with skip → Output
```
Where r << d (bottleneck), so parameters per adapter = 2×d×r.

Benefits: Multiple adapters for multiple tasks can be swapped at inference time without reloading the base model — useful for multi-task serving. One base model, many task-specific adapters loaded on demand.

Limitation: Slight inference overhead (extra layer per Transformer block). LoRA is generally preferred now because it has zero inference overhead after merging.


### Q7. What is RLHF, and how is it used to align LLMs?
**A:** RLHF (Reinforcement Learning from Human Feedback) aligns LLMs with human values and preferences.

**Three phases:**
1. **SFT (Supervised Fine-Tuning):** Fine-tune base model on high-quality human-written demonstrations
2. **Reward Model Training:** Humans compare pairs of model outputs (A vs B); train a reward model to predict human preference scores
3. **RL Optimisation (PPO):** Use PPO to fine-tune the SFT model to maximise reward model scores, while adding KL-divergence penalty to prevent drifting too far from the SFT model

This is how ChatGPT, Claude, and all chat models were made safe, helpful, and conversational. The reward model approximates "what would a human rate highest?"


### Q8. What is instruction tuning, and why is it important for chat models?
**A:** Instruction tuning = fine-tuning a base LLM on a large dataset of (instruction, response) pairs where the model learns to follow instructions across many tasks.

Before instruction tuning: base models complete text (GPT-3 would continue "What is the capital of France?" with more questions, not with "Paris").

After instruction tuning: models follow instructions ("Summarise this text in 3 bullet points", "Translate to Spanish", "Write a poem about X") reliably.

Why critical: All practical LLM applications require instruction-following. Without it, base models are too unpredictable to deploy.

Key datasets: FLAN (Google), Alpaca, ShareGPT, Open Assistant. Modern open models (Llama 3 Instruct, Mistral Instruct) are all instruction-tuned.


### Q9. How do you prepare a dataset for fine-tuning an LLM?
**A:**
**Data requirements:**
- Minimum: 500-1000 high-quality examples. Quality >> quantity.
- Format: (instruction, input, output) triplets or chat format (system, user, assistant)
- Diversity: Cover all use cases, edge cases, and failure modes

**Data pipeline:**
1. **Collection**: Curate from existing data, use GPT-4 to generate synthetic pairs, human annotators
2. **Cleaning**: Remove duplicates, low-quality examples, PII, harmful content
3. **Formatting**: Convert to target format (Alpaca, ShareGPT, ChatML)
4. **Validation**: Sample 100 examples manually; review quality
5. **Split**: 90% train, 5% validation, 5% test
6. **Tokenisation**: Tokenise with the target model's tokeniser; check length distribution

Common format (Alpaca):
```json
{"instruction": "Explain bail under BNS", "input": "", "output": "Under BNS Section 437..."}
```


### Q10. What is catastrophic forgetting, and how do you prevent it during fine-tuning?
**A:** Catastrophic forgetting = when fine-tuning on a new task, the model "forgets" (overwrites weights for) previously learned knowledge.

Example: Fine-tune Llama on legal QA → it becomes great at legal questions but loses its ability to write code or answer general knowledge questions.

**Prevention:**
1. **LoRA/PEFT** — only train small adapter weights; base model knowledge stays intact
2. **Replay** — mix original pre-training data with new task data during fine-tuning
3. **Elastic Weight Consolidation (EWC)** — regularise against changing weights important for old tasks
4. **Low learning rate** — fine-tune with very small LR to make small updates to important weights
5. **Gradient clipping** — prevent large gradient updates that could catastrophically overwrite weights
6. **Evaluation on general benchmarks** — monitor MMLU and general QA metrics during fine-tuning; stop if they drop too much


### Q11. When should you choose fine-tuning over RAG over prompt engineering?
**A:**
**Use Prompt Engineering when:** Simple adjustments to behaviour, quick prototyping, limited data, or knowledge/context can be injected in the prompt.

**Use RAG when:** The task requires factual knowledge that: (1) changes frequently, (2) is domain-specific and large, (3) needs citations, (4) exceeds context window. Cheaper and faster than fine-tuning.

**Use Fine-tuning when:**
- Consistent style/format/persona that prompts can't achieve reliably
- Reducing inference cost by making a smaller model work as well as a large prompted model
- Domain-specific linguistic patterns (legal citations always formatted a certain way)
- Very high-volume task where per-token API cost of long prompts is prohibitive

**Best combination:** RAG for knowledge grounding + Fine-tuning for style/format/persona. Use them together, not as either/or.


### Q12. How do you evaluate a fine-tuned model's performance?
**A:**
**Automatic metrics:**
- Task-specific: accuracy, F1, BLEU/ROUGE for text generation
- Perplexity on a held-out validation set
- LLM-as-judge: GPT-4 rates outputs 1-5 on quality dimensions

**Human evaluation:**
- Side-by-side comparison: base model vs fine-tuned model output for same prompts
- Rate on: accuracy, coherence, helpfulness, safety

**Regression testing:**
- Run on a golden test set pre-built before fine-tuning
- Ensure fine-tuned model scores higher on target task AND doesn't regress on general benchmarks (MMLU, etc.)

**Business metrics (production):**
- User satisfaction scores, task completion rates, escalation rate (for support agents)
- A/B test: route 5% traffic to fine-tuned model; compare engagement metrics vs baseline


### Q13. What is synthetic data generation, and how do you use it for fine-tuning?
**A:** Synthetic data = AI-generated training examples. Use a powerful frontier model (GPT-4) to generate (instruction, response) pairs that you then use to fine-tune a smaller model.

**Pipeline:**
1. Write a seed set of 50-100 human-written examples (diverse, high quality)
2. GPT-4 generates variations and new examples from seeds (Self-Instruct approach)
3. Quality filter: GPT-4 scores its own outputs; keep only high-quality ones
4. Deduplication: remove near-duplicate examples
5. Human spot-check: review 5-10% of generated data for quality and safety
6. Use for fine-tuning the smaller target model

**Examples:** Alpaca (Stanford) used GPT-3.5 to generate 52K examples → fine-tuned Llama-7B. WizardLM evol-instruct systematically creates harder examples.

**Legal consideration:** Check the LLM provider's ToS — many prohibit using API outputs to train competing models.


### Q14. What are the key hyperparameters for fine-tuning?
**A:**
| Hyperparameter | Typical Range | Effect |
|---------------|--------------|--------|
| **Learning rate** | 1e-5 to 3e-4 | Most important. Too high = catastrophic forgetting. Too low = no learning. Use linear warmup + cosine decay. |
| **Batch size** | 4-64 (effective) | Larger = more stable gradients. Use gradient accumulation to simulate large batches. |
| **Epochs** | 1-5 | 1-3 for large datasets. More = risk of overfitting. |
| **LoRA rank (r)** | 8-64 | Higher = more expressive but more params. Start with r=16. |
| **LoRA alpha** | 16-32 | Scaling factor. Usually set to 2×r. |
| **LoRA dropout** | 0.0-0.1 | Regularisation. 0.05 is safe default. |
| **Max sequence length** | 512-4096 | Match your data; longer = more memory |

Always use a small learning rate warmup (5-10% of steps) to avoid large early updates.


### Q15. How do you fine-tune a model for a specific domain (legal, medical, finance)?
**A:**
**Two-stage approach:**
1. **Domain-adaptive continued pre-training** (optional but powerful): Continue pre-training on large unlabelled domain corpus (legal texts, medical papers) — the model learns domain vocabulary and patterns
2. **Task-specific fine-tuning**: Fine-tune on labelled (instruction, response) pairs for your specific tasks

**Dataset sources:**
- Legal: case law, statutes, legal QA pairs
- Medical: PubMed, clinical notes (de-identified), medical QA benchmarks
- Finance: SEC filings, earnings calls, financial QA

**Special considerations:**
- Domain tokenisation: medical/legal terms may tokenise poorly — consider vocabulary extension
- Hallucination risk is higher in specialised domains — train on factual, well-sourced data only
- Evaluation needs domain experts — general benchmarks don't assess domain quality

🏷️ *Nyaya-Pro's next evolution: continued pre-training Llama 3 8B on Indian legal corpus (BNS, BNSS, Constitution text) before task-specific LoRA fine-tuning for legal QA.*


### Q16. What is continual pre-training, and when would you use it?
**A:** Continual pre-training = taking a pre-trained model and continuing its pre-training (next-token prediction on raw text) on a domain-specific corpus, before any task-specific fine-tuning.

**When to use:**
- Your domain has highly specialised vocabulary the base model doesn't know well (rare legal citations, medical drug names, proprietary technical jargon)
- You have a very large unlabelled domain corpus (millions of documents)
- You need the model to understand domain-specific writing style and structure

**vs fine-tuning:** Continual pre-training is unsupervised (raw text, no labels). Fine-tuning is supervised (instruction-response pairs). They're complementary: continual pre-train first, then fine-tune for task.

**Risk:** Still prone to catastrophic forgetting of general knowledge. Mitigate by mixing 5-10% general data into the domain corpus.


### Q17. How do you merge multiple LoRA adapters?
**A:** Multiple LoRA adapters (trained for different tasks) can be merged into one model.

**Methods:**
1. **Sequential merge**: Apply one adapter's weight updates after another. Simple but quality degrades if adapters conflict.
2. **Linear merge (averaging)**: `ΔW_merged = α₁×ΔW₁ + α₂×ΔW₂`. The α weights control each adapter's contribution. Works when adapters are for complementary tasks.
3. **TIES (Task-specific LoRA merging)**: More sophisticated — prune redundant parameters, resolve sign conflicts between adapters.
4. **DARE (Drop and Rescale)**: Randomly drop adapter parameters before merging to reduce interference.

Tools: `peft` library supports adapter merging. `mergekit` is a popular open-source tool for model merging.

Use case: Merge a "legal reasoning" adapter + a "citation formatting" adapter + a "document summarisation" adapter into one legal AI model.


### Q18. What is the difference between SFT and alignment training?
**A:**
**SFT (Supervised Fine-Tuning):**
- Goal: Teach the model to follow instructions correctly
- Data: (instruction, ideal response) pairs — human-written demonstrations
- Training: Standard cross-entropy on the response tokens
- Outcome: Model can follow instructions; may still be biased, unsafe, or sycophantic

**Alignment Training (RLHF/DPO/RLAIF):**
- Goal: Make the model helpful, harmless, and honest (HHH)
- Data: Human preference pairs — which of these two responses is better?
- Training: Reward signal or direct preference optimisation
- Outcome: Model prefers helpful, truthful, safe responses over technically correct but harmful ones

**Pipeline:** SFT → Alignment. SFT creates the instruction-following foundation; alignment adds values and preferences on top.


### Q19. What is RLAIF (RL from AI Feedback), and how does it differ from RLHF?
**A:**
**RLHF:** Humans compare model outputs and label preferences → reward model trained on human labels → PPO optimisation.

**Rlaif:** An AI model (Claude, GPT-4) compares outputs and generates preference labels → same reward model training → same PPO/DPO.

**Why RLAIF:**
- Human labelling is expensive, slow, inconsistent
- AI labelling is cheap, fast, infinitely scalable
- Constitutional AI (Claude) uses RLAIF — the AI critiques its own outputs against constitutional principles

**Trade-offs:**
- RLAIF can perpetuate AI biases from the judge model
- Human preferences capture things AI can't (cultural nuance, emotional resonance)
- Best approach: RLAIF for scale + periodic human audits for quality control

**Debate:** RLAIF quality depends on the quality of the AI judge. Using GPT-4 to label data for a GPT-4 competitor is now common practice (and somewhat controversial).


### Q20. What is knowledge distillation for fine-tuning, and what are the legal considerations?
**A:** Training a small model to mimic a large model's outputs — specifically for fine-tuning purposes, using API-generated responses as training data.

**Process:** Call GPT-4 API with your prompts → collect high-quality responses → use as fine-tuning data for a smaller open-source model (Llama, Mistral).

**Legal considerations — CRITICAL:**
- **OpenAI ToS** explicitly prohibits: "You may not use output from the Services to develop models that compete with OpenAI."
- **Anthropic** has similar restrictions on Claude outputs
- **Google Gemini** also prohibits competitive model training from API outputs

**Legal approaches:**
- Use open models as the teacher (Llama 3 70B → distil to Llama 3 8B) — no ToS issues
- License agreements: some providers offer data licensing for enterprise customers
- Use human-generated data or your own model as the teacher

🏷️ *For Nyaya-Pro fine-tuning: I would use Llama 3 70B (open-source teacher) to generate legal QA pairs, then fine-tune Llama 3 8B (student) — fully legal, no ToS issues.*


### Q21. Your fine-tuned LLM produces factually wrong outputs due to training data quality issues. How do you fix it?
**A:**
1. **Audit training data** — sample 100 examples and manually verify factual accuracy. Find and remove errors.
2. **Source filtering** — trace each training example to its source; remove low-quality sources
3. **Data quality scoring** — use an LLM to rate factual accuracy of each training example (1-5); filter out low scores
4. **Reduce learning rate** — if the model is memorising noise in the data rather than learning patterns
5. **RAG for facts** — don't try to bake frequently-changing facts into model weights; use RAG for fact retrieval
6. **Add factual consistency regulariser** — during fine-tuning, penalise outputs that contradict known factual claims (requires a factual consistency model)
7. **Post-training fact-checking** — add a fact-checker layer that validates outputs against a knowledge base before returning to users


### Q22. You must choose between LoRA and full fine-tuning for a domain-specific assistant. How do you decide?
**A:**
**Choose LoRA when:**
- Limited GPU budget (< 4× A100s)
- Need quick iteration cycles (LoRA is much faster to experiment with)
- Task is mainly behaviour/format adaptation (not deep knowledge)
- Need multiple task-specific adapters on one base model
- Want to preserve full general capabilities (LoRA is much safer for catastrophic forgetting)

**Choose full fine-tuning when:**
- Highest quality is essential and budget allows (medical diagnosis, legal advice where accuracy is safety-critical)
- Deep domain adaptation required — the model needs to learn completely different knowledge patterns
- Have access to large GPU cluster (8×A100 or more)
- One-time training with no need for multi-task capability

**Practical answer:** Start with LoRA. If quality is insufficient after tuning hyperparameters and improving data quality, then consider full fine-tuning. In 95% of cases, QLoRA achieves production-quality results.


### Q23. Your fine-tuned model memorized training data verbatim instead of learning patterns. How do you fix overfitting?
**A:**
1. **More diverse training data** — duplicated examples cause memorisation; diversify your dataset
2. **Deduplication** — remove near-duplicate training examples (use MinHash or embedding similarity)
3. **Lower learning rate** — high LR drives the model to memorise training examples
4. **Fewer epochs** — 1-2 epochs often sufficient; 5+ epochs → overfitting
5. **Dropout** — enable dropout during fine-tuning (especially for LoRA: LoRA dropout)
6. **Weight decay / L2 regularisation** — penalises large parameter magnitudes
7. **Early stopping** — monitor validation loss; stop when it starts increasing
8. **Data augmentation** — paraphrase training examples to create varied versions of the same concept
9. **Increase data size** — if you can't get more diverse real examples, use synthetic data generation


### Q24. Your fine-tuned LLM forgot its general capabilities after domain fine-tuning. How do you fix catastrophic forgetting?
**A:**
1. **Use LoRA / PEFT** — the #1 fix. Base model weights are frozen; only adapters change. General capabilities stay intact.
2. **Mix general data** — include 10-20% general-purpose examples in your fine-tuning dataset alongside domain data
3. **EWC (Elastic Weight Consolidation)** — penalise updates to weights that were important for previous tasks
4. **Lower learning rate** — smaller updates = less overwriting of existing knowledge
5. **Shorter training** — fine-tune for fewer steps; stop before general capabilities degrade
6. **Monitor general benchmarks** — track MMLU, TruthfulQA, and coding benchmarks during fine-tuning; implement early stopping when they drop
7. **Multi-task fine-tuning** — fine-tune simultaneously on domain task AND a sample of general tasks

If using LoRA from the start, forgetting is minimal. If you used full fine-tuning and forgot capabilities, the safest fix is switching to LoRA.


### Q25. Your RLHF preference data has low annotator agreement. How do you ensure data quality?
**A:**
1. **Clear annotation guidelines** — detailed rubrics defining "helpful", "harmless", "honest" with concrete examples and edge cases. Ambiguous guidelines → disagreement.
2. **Annotator training** — calibration sessions where annotators align on edge cases before production annotation
3. **Inter-annotator agreement metric** — measure Cohen's Kappa; target > 0.6. Investigate and address sources of disagreement.
4. **Majority voting** — for each pair, use 3-5 annotators; take majority vote. Disagreements highlight genuinely ambiguous cases.
5. **Disagreement filtering** — for very low-agreement examples (all three annotators disagree), consider removing them from training data
6. **Expert review** — have domain experts (lawyers, doctors) review domain-specific preference pairs
7. **Task decomposition** — break "which is better" into specific dimensions (accuracy, safety, helpfulness) rated separately — reduces ambiguity
